In [1]:
from google.colab import files
files.upload()

Saving modelling_after_pp_V2.csv to modelling_after_pp_V2 (1).csv


In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [3]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/convert_slow_tokenizer.py:560: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.


In [4]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

In [5]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [6]:
%pip install datasets
from datasets import *

ds = load_dataset("csv", data_files="/content/modelling_after_pp_V2.csv")

train_testvalid = ds['train'].train_test_split(test_size=0.2)
# Split the 10% test + valid in half test, half valid
test_valid = train_testvalid['test'].train_test_split(test_size=0.5)
# gather everyone if you want to have a single DatasetDict
ds = DatasetDict({
    'train': train_testvalid['train'],
    'test': test_valid['test'],
    'valid': test_valid['train']})

tokenized_imdb = ds.map(preprocess_function, batched=True)

Map:   0%|          | 0/295768 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/36971 [00:00<?, ? examples/s]

Map:   0%|          | 0/36971 [00:00<?, ? examples/s]

In [7]:
%pip install evaluate
import evaluate

accuracy = evaluate.load("accuracy")

In [8]:
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [9]:
id2label ={
    0: 'sadness',
    1: 'joy',
    2: 'love',
    3: 'anger',
    4: 'fear',
    5: 'surprise'
}

label2id = {"sadness": 0, "joy": 1, "love": 2, "anger": 3, "fear": 4, "surprise": 5}

In [10]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base", num_labels=6, id2label=id2label, label2id=label2id
)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [11]:
%pip uninstall -y accelerate
%pip uninstall -y transformers
%pip install -U accelerate
%pip install -U transformers
training_args = TrainingArguments(
    output_dir="my_awesome_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_imdb["train"],
    eval_dataset=tokenized_imdb["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Found existing installation: accelerate 0.30.1
Uninstalling accelerate-0.30.1:
  Successfully uninstalled accelerate-0.30.1
Found existing installation: transformers 4.41.1
Uninstalling transformers-4.41.1:
  Successfully uninstalled transformers-4.41.1
  Using cached accelerate-0.30.1-py3-none-any.whl (302 kB)


  Using cached transformers-4.41.1-py3-none-any.whl (9.1 MB)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.018800,0.018485,0.995212


Epoch,Training Loss,Validation Loss,Accuracy
1,0.018800,0.018485,0.995212
2,0.010500,0.013245,0.996403


TrainOutput(global_step=36972, training_loss=0.03626031601156854, metrics={'train_runtime': 9551.7518, 'train_samples_per_second': 61.93, 'train_steps_per_second': 3.871, 'total_flos': 1.4083555571862336e+16, 'train_loss': 0.03626031601156854, 'epoch': 2.0})

In [12]:
from transformers import pipeline
text = ["When i think about school i feel depressed", "finally done with finals", "i feel loved", "i cant believe they did that", "so scared about today", "stunned by the exam today"]
classifier = pipeline("sentiment-analysis", model="/content/my_awesome_model")

for t in text:
  print(classifier(t))

[{'label': 'sadness', 'score': 0.999991774559021}]
[{'label': 'joy', 'score': 0.9925570487976074}]
[{'label': 'love', 'score': 0.9999827146530151}]
[{'label': 'anger', 'score': 0.3374536335468292}]
[{'label': 'fear', 'score': 0.9998224377632141}]
[{'label': 'surprise', 'score': 0.5630893111228943}]


In [13]:
text = "so bad that it is good"

classifier(text)

[{'label': 'joy', 'score': 0.4398166537284851}]

In [14]:
text = "so bad that it is actually good"

classifier(text)

[{'label': 'anger', 'score': 0.3712025284767151}]

In [15]:
text = "expected it to be bad and still disappointed"

classifier(text)

[{'label': 'sadness', 'score': 0.9250255227088928}]

In [16]:
text = "expected it to be bad but was amazing"

classifier(text)

[{'label': 'joy', 'score': 0.9413429498672485}]

In [17]:
text = "Three years later, i still feel sad"

classifier(text)

[{'label': 'sadness', 'score': 0.9999914169311523}]

In [18]:
text = "Three years later, i feel good again"

classifier(text)

[{'label': 'joy', 'score': 0.9999922513961792}]

In [21]:
import torch

torch.save(model.state_dict(), 'deberta_model.pth')